# OmniSupport AI — Llama 3.1 Fine-tuning + RAG + Side-by-Side Demo

Run this notebook top to bottom in **Colab with GPU enabled**.

It will:
- download the customer support dataset automatically,
- prepare clean instruction/response pairs,
- fine-tune **Meta-Llama-3.1-8B-Instruct** with LoRA,
- build a lighter RAG knowledge base that shares only metadata and short guidance,
- compare **base model vs fine-tuned model** both **without RAG** and with **light RAG**.

You only need to provide your **Hugging Face token** for gated model access.


In [ ]:
!pip -q install --upgrade transformers accelerate bitsandbytes peft datasets trl faiss-cpu sentence-transformers kagglehub

## 1) Hugging Face authentication

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if not hf_token:
    hf_token = getpass("Paste your Hugging Face token: ")

login(token=hf_token)
print("Hugging Face login completed.")

Paste your Hugging Face token: ··········
Hugging Face login completed.


## 2) Imports and global configuration

In [ ]:
import gc
import html
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from IPython.display import display, HTML

from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from sentence_transformers import SentenceTransformer
import faiss
import kagglehub

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer

assert torch.cuda.is_available(), "Enable GPU runtime in Colab before running this notebook."

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
DATASET_ID = "suraj520/customer-support-ticket-dataset"

BASE_DIR = Path("/content")
DRIVE_DIR = BASE_DIR / "drive" / "MyDrive"
ADAPTER_DIR = DRIVE_DIR / "OmniSupport_Llama3_Adapter"
CHECKPOINT_DIR = DRIVE_DIR / "OmniSupport_Checkpoints"

MAX_TRAIN_EXAMPLES = 1200
MAX_RAG_DOCS = 800
RAG_TOP_K = 1
RAG_CONTEXT_CHAR_LIMIT = 500

SYSTEM_PROMPT = (
    "You are OmniSupport AI, a customer support assistant. "
    "Give clear, constructive, step-by-step help in a warm professional tone. "
    "Acknowledge the issue, use a consistent support structure, explain the next actions, "
    "and mention what to do if the problem persists. "
    "Keep the answer concise, practical, and easy to follow."
)

torch.backends.cuda.matmul.allow_tf32 = True
print("Configuration ready.")


Configuration ready.


In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 3) Load tokenizer and base model

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID,token=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=True,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False

print("Base model loaded.")

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Base model loaded.


## 4) Download and clean the support dataset automatically

In [ ]:
def load_support_dataframe() -> pd.DataFrame:
    local_csv = BASE_DIR / "customer_support_tickets.csv"
    if local_csv.exists():
        return pd.read_csv(local_csv)

    dataset_path = Path(kagglehub.dataset_download(DATASET_ID))
    csv_files = list(dataset_path.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV file found inside downloaded dataset folder: {dataset_path}")
    return pd.read_csv(csv_files[0])

df = load_support_dataframe()
print("Loaded rows:", len(df))
print("Columns:", list(df.columns))
df.head(3)

100%|██████████| 828k/828k [00:00<00:00, 67.9MB/s]

Extracting files...
Loaded rows: 8469
Columns: ['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0


In [ ]:
def pick_col(df, candidates):
    lookup = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().strip()
        if key in lookup:
            return lookup[key]
    for col in df.columns:
        col_l = col.lower().strip()
        if any(token.lower() in col_l for token in candidates):
            return col
    return None

COL_DESCRIPTION = pick_col(df, ["Ticket Description", "description", "issue"])
COL_SUBJECT = pick_col(df, ["Ticket Subject", "subject"])
COL_TYPE = pick_col(df, ["Ticket Type", "type"])
COL_PRIORITY = pick_col(df, ["Ticket Priority", "priority"])
COL_PRODUCT = pick_col(df, ["Product Purchased", "product"])
COL_STATUS = pick_col(df, ["Ticket Status", "status"])
COL_CHANNEL = pick_col(df, ["Ticket Channel", "channel"])

required = [COL_DESCRIPTION, COL_SUBJECT, COL_TYPE, COL_PRIORITY, COL_PRODUCT, COL_STATUS, COL_CHANNEL]
if any(c is None for c in required):
    raise ValueError(f"Missing expected columns. Found mapping: {required}")

def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def classify_issue(subject: str, ticket_type: str, description: str) -> str:
    text = f"{subject} {ticket_type} {description}".lower()
    if any(k in text for k in ["password", "login", "sign in", "account access", "2fa", "verification", "otp"]):
        return "account_access"
    if any(k in text for k in ["refund", "billing", "invoice", "payment", "charged", "subscription", "cancel"]):
        return "billing"
    if any(k in text for k in ["data loss", "lost", "deleted", "recover", "backup"]):
        return "data_loss"
    if any(k in text for k in ["delivery", "shipping", "order", "tracking", "arrive"]):
        return "delivery"
    if any(k in text for k in ["setup", "install", "installation", "activation", "pair"]):
        return "setup_install"
    if any(k in text for k in ["network", "wifi", "internet", "connect", "latency", "slow"]):
        return "connectivity"
    if any(k in text for k in ["hardware", "screen", "battery", "fan", "overheat", "broken", "crack", "no power"]):
        return "hardware"
    if any(k in text for k in ["software", "app", "crash", "error", "bug", "update", "compatibility"]):
        return "software"
    return "general"

def build_instructions(row):
    product = safe_str(row[COL_PRODUCT]) or "the product"
    subject = safe_str(row[COL_SUBJECT])
    ticket_type = safe_str(row[COL_TYPE])
    priority = safe_str(row[COL_PRIORITY])
    channel = safe_str(row[COL_CHANNEL])
    status = safe_str(row[COL_STATUS])
    description = safe_str(row[COL_DESCRIPTION]).replace("{product_purchased}", product)

    issue_class = classify_issue(subject, ticket_type, description)
    urgency = priority.lower()
    if any(token in urgency for token in ["high", "urgent", "critical"]):
        opener = "I understand this is urgent and frustrating."
    else:
        opener = "I understand the issue and I will help you work through it."

    instruction = (
        f"Customer support request about {product}. "
        f"Ticket subject: {subject}. "
        f"Issue type: {ticket_type}. "
        f"Channel: {channel}. "
        f"Priority: {priority}. "
        f"Status: {status}. "
        f"Customer message: {description}"
    )

    if issue_class == "account_access":
        steps = [
            "Confirm the email or username used for the account.",
            "Reset the password and ask the customer to check spam or junk folders for the reset link.",
            "Ask the customer to clear browser cache or try another browser or device.",
            "If two-factor authentication is enabled, verify the code or backup method.",
        ]
        escalation = "If the issue continues, request the exact error message and escalate to account support."
    elif issue_class == "billing":
        steps = [
            "Verify the invoice, transaction date, and payment method.",
            "Check whether the charge was pending, duplicated, or already refunded.",
            "Ask the customer to confirm the last four digits of the card or the billing email.",
            "If needed, route the case to billing review with the transaction ID.",
        ]
        escalation = "If the issue is unresolved, escalate to billing support with the order or invoice number."
    elif issue_class == "data_loss":
        steps = [
            "Stop using the device or app immediately to avoid overwriting recoverable data.",
            "Check whether a cloud backup, version history, or recycle bin is available.",
            "Confirm when the data was last visible and whether any recent update occurred.",
            "Escalate to recovery support if the data is mission-critical.",
        ]
        escalation = "Ask for the affected file names, last known location, and a screenshot if possible."
    elif issue_class == "delivery":
        steps = [
            "Check the order status and tracking number.",
            "Confirm the shipping address and expected delivery date.",
            "Ask the customer to wait for the latest carrier scan if the order is still in transit.",
            "If the package is delayed or lost, create a logistics escalation.",
        ]
        escalation = "Request the order ID and tracking number if the customer wants a follow-up."
    elif issue_class == "setup_install":
        steps = [
            "Verify the device or system requirements for the product.",
            "Re-run the setup or installation using administrator permissions if needed.",
            "Confirm the latest version is installed and restart the device after setup.",
            "If activation fails, request the exact error code and license details.",
        ]
        escalation = "If the installation still fails, escalate with the error code and environment details."
    elif issue_class == "connectivity":
        steps = [
            "Ask the customer to test a stable internet connection or switch networks.",
            "Restart the router, app, or device to clear temporary connection issues.",
            "Check whether VPN, firewall, or proxy settings are blocking access.",
            "If the issue remains, collect logs and the exact time of failure.",
        ]
        escalation = "Escalate to network support if the error repeats after a clean reconnect."
    elif issue_class == "hardware":
        steps = [
            "Ask the customer to power-cycle the device and inspect visible damage.",
            "Check cables, ports, battery level, and overheating symptoms.",
            "Recommend using the latest firmware or drivers if the product supports them.",
            "If the hardware appears damaged, schedule service or replacement review.",
        ]
        escalation = "Request photos, serial number, and the exact symptoms before escalation."
    elif issue_class == "software":
        steps = [
            "Confirm the app or software version and whether the issue started after an update.",
            "Restart the application and try the latest stable update.",
            "Clear temporary files or reinstall only if necessary.",
            "Collect the error code, screenshot, and steps to reproduce the problem.",
        ]
        escalation = "Escalate to technical support if the issue persists after the basic checks."
    else:
        steps = [
            "Acknowledge the problem and restate the main issue clearly.",
            "Ask for the most relevant missing detail if the case is incomplete.",
            "Give the customer a short, direct set of next steps to try.",
            "Offer escalation if the steps do not resolve the problem.",
        ]
        escalation = "Collect the error message, product version, and screenshot if the issue continues."

    response = (
        f"{opener}\n\n"
        f"Problem summary: Customer reported a {issue_class.replace('_', ' ')} issue on {product}.\n"
        f"Diagnosis: This ticket should follow a structured support workflow.\n"
        f"Troubleshooting steps:\n"
        + "\n".join([f"{i+1}. {step}" for i, step in enumerate(steps)])
        + f"\n\nEscalation decision: {escalation}\n"
        + f"Closing: Please share the error message, screenshots, or logs if the issue persists."
    )

    return instruction, response, issue_class

records = []
for _, row in df.head(MAX_TRAIN_EXAMPLES).iterrows():
    instruction, response, issue_class = build_instructions(row)
    records.append({
        "instruction": instruction,
        "response": response,
        "category": issue_class,
    })

train_df = pd.DataFrame(records)
train_df.head(3)


,instruction,response,category
0,Customer support request about GoPro Hero. Tic...,I understand this is urgent and frustrating.\n...,billing
1,Customer support request about LG Smart TV. Ti...,I understand this is urgent and frustrating.\n...,software
2,Customer support request about Dell XPS. Ticke...,I understand the issue and I will help you wor...,connectivity


## 5) Build chat-formatted training text

In [ ]:
def to_chat_text(instruction: str, response: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": response},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_df["text"] = train_df.apply(lambda r: to_chat_text(r["instruction"], r["response"]), axis=1)
dataset = Dataset.from_pandas(train_df[["text", "category"]], preserve_index=False)

print("Training examples:", len(dataset))
print(dataset[0]["text"][:1000])

Training examples: 1200
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are OmniSupport AI, a customer support assistant. Give clear, constructive, step-by-step help in a warm professional tone. Acknowledge the issue, use a consistent support structure, explain the next actions, and mention what to do if the problem persists. Keep the answer concise, practical, and easy to follow.<|eot_id|><|start_header_id|>user<|end_header_id|>

Customer support request about GoPro Hero. Ticket subject: Product setup. Issue type: Technical issue. Channel: Social media. Priority: Critical. Status: Pending Customer Response. Customer message: I'm having an issue with the GoPro Hero. Please assist.

Your billing zip code is: 71701.

We appreciate that you have requested a website address.

Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I understand this

## 6) Prepare LoRA fine-tuning

In [ ]:
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

train_model = get_peft_model(base_model, lora_config)
train_model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,
    bf16=False,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

from trl import SFTTrainer

trainer = SFTTrainer(
    model=train_model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Trainer ready.")


trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


Adding EOS to train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

Trainer ready.


## 7) Train and save the adapter

In [ ]:
train_result = trainer.train()
print(train_result)

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
train_model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f"Saved fine-tuned adapter to: {ADAPTER_DIR}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.


Step,Training Loss
10,3.270382
20,2.014416
30,0.912628
40,0.641011
50,0.627862
60,0.533001
70,0.476872
80,0.439211
90,0.445760
100,0.404532


Step,Training Loss
10,3.270382
20,2.014416
30,0.912628
40,0.641011
50,0.627862
60,0.533001
70,0.476872
80,0.439211
90,0.445760
100,0.404532


TrainOutput(global_step=900, training_loss=0.3538954422208998, metrics={'train_runtime': 8021.8965, 'train_samples_per_second': 0.449, 'train_steps_per_second': 0.112, 'total_flos': 5.180919477623194e+16, 'train_loss': 0.3538954422208998})
Saved fine-tuned adapter to: /content/drive/MyDrive/OmniSupport_Llama3_Adapter


## 8) Build the RAG knowledge base

In [ ]:
SUPPORT_PLAYBOOK = {
    "account_access": (
        "Account access troubleshooting: confirm the registered email, reset the password, "
        "check spam or junk folders for reset links, clear browser cache, disable VPN temporarily, "
        "and verify 2FA or backup codes before escalation."
    ),
    "billing": (
        "Billing troubleshooting: verify invoice number, transaction date, payment method, "
        "pending vs completed charge, refund status, and escalate with transaction ID if needed."
    ),
    "data_loss": (
        "Data loss troubleshooting: stop overwriting the device, check cloud backup, version history, "
        "trash or recycle bin, recent sync activity, and escalate recovery if the file is critical."
    ),
    "delivery": (
        "Delivery troubleshooting: confirm order ID, tracking number, shipping address, carrier scan, "
        "estimated delivery date, and create logistics escalation if the parcel is delayed or missing."
    ),
    "setup_install": (
        "Setup and installation troubleshooting: verify system requirements, reinstall or re-run setup, "
        "restart after installation, ensure the latest version is used, and collect the exact error code."
    ),
    "connectivity": (
        "Connectivity troubleshooting: test a stable network, restart router/device, disable VPN or proxy, "
        "check firewall rules, and collect the time of failure and error message."
    ),
    "hardware": (
        "Hardware troubleshooting: power-cycle the device, inspect cables and ports, check overheating, "
        "update firmware or drivers when relevant, and escalate with serial number and photos if damaged."
    ),
    "software": (
        "Software troubleshooting: confirm app version, restart the app, install the latest update, "
        "clear temporary files, reproduce the issue, and collect screenshot plus error code."
    ),
    "general": (
        "General support workflow: acknowledge the issue, restate the problem clearly, give concise next steps, "
        "ask for missing details, and escalate with logs or screenshots if unresolved."
    ),
}

rag_docs = []
for _, row in train_df.sample(min(MAX_RAG_DOCS, len(train_df)), random_state=42).iterrows():
    cat = row["category"]
    metadata = row["instruction"].split("Customer message:", 1)[0].strip()
    rag_docs.append(
        f"Category: {cat}. Metadata: {metadata}. Guidance: {SUPPORT_PLAYBOOK.get(cat, SUPPORT_PLAYBOOK['general'])}"
    )

for cat, guide in SUPPORT_PLAYBOOK.items():
    rag_docs.append(f"Category: {cat}. Guidance: {guide}")

embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(rag_docs, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)

rag_index = faiss.IndexFlatIP(doc_embeddings.shape[1])
rag_index.add(doc_embeddings)

print("RAG documents:", len(rag_docs))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

RAG documents: 809


In [ ]:
def retrieve_context(query: str, top_k: int = RAG_TOP_K) -> str:
    q_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = rag_index.search(q_emb, top_k)
    matches = [rag_docs[i] for i in indices[0] if i != -1]
    context = "\n\n".join(matches)
    if len(context) > RAG_CONTEXT_CHAR_LIMIT:
        context = context[:RAG_CONTEXT_CHAR_LIMIT].rsplit(" ", 1)[0] + "..."
    return context


def build_inference_prompt(user_query: str, context: str = "", use_rag: bool = False) -> str:
    if use_rag and context:
        user_message = (
            "Use the retrieved support notes only as light context. Do not copy them verbatim.\n\n"
            f"Retrieved support notes:\n{context}\n\n"
            f"Customer question:\n{user_query}\n\n"
            "Answer with a warm support tone, a short structured workflow, and practical next steps."
        )
    else:
        user_message = (
            "Answer the customer question directly without retrieved context.\n\n"
            f"Customer question:\n{user_query}\n\n"
            "Answer with a warm support tone, a short structured workflow, and practical next steps."
        )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

GEN_KWARGS = {
    "max_new_tokens": 180,
    "do_sample": False,
    "temperature": 0.0,
    "top_p": 1.0,
    "repetition_penalty": 1.08,
    "pad_token_id": tokenizer.eos_token_id,
    "eos_token_id": tokenizer.eos_token_id,
}

def generate_response(model, prompt: str) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, **GEN_KWARGS)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Inference helpers ready.")


## 9) Side-by-side comparison: base vs fine-tuned

This section now runs two checks: first **without RAG** to measure the fine-tuning effect directly, and then with **light RAG** to see how retrieval changes the answer.


In [ ]:
def load_base_inference_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        attn_implementation="eager",
    )
    model.config.use_cache = False
    model.eval()
    return model


def load_tuned_inference_model():
    base_for_tuned = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        attn_implementation="eager",
    )
    tuned = PeftModel.from_pretrained(base_for_tuned, str(ADAPTER_DIR))
    tuned.eval()
    return tuned


def _render_comparison_card(title: str, retrieved_context: str, left_title: str, left_text: str, right_title: str, right_text: str):
    safe_context = html.escape(retrieved_context if retrieved_context else "(RAG disabled for this evaluation)").replace("\n", "<br>")
    safe_left = html.escape(left_text).replace("\n", "<br>")
    safe_right = html.escape(right_text).replace("\n", "<br>")

    # Using higher contrast colors (darker text and slightly more saturated borders/backgrounds)
    display(HTML(f'''
    <div style="margin:10px 0 6px 0; font-size:16px; font-weight:700; color: #333;">{html.escape(title)}</div>
    <div style="display:flex; gap:16px; align-items:stretch; width:100%; margin-bottom:18px;">
      <div style="flex:1; border:2px solid #b71c1c; border-radius:12px; padding:14px; background:#ffebee; color: #1a1a1a;">
        <h3 style="margin-top:0; color: #b71c1c;">{html.escape(left_title)}</h3>
        <div style="font-size:13px; color:#444; margin-bottom:8px; border-bottom: 1px solid #ffcdd2; padding-bottom: 5px;"><b>Retrieved context:</b><br>{safe_context}</div>
        <div style="white-space:normal; line-height:1.5; font-weight: 400;">{safe_left}</div>
      </div>
      <div style="flex:1; border:2px solid #2e7d32; border-radius:12px; padding:14px; background:#e8f5e9; color: #1a1a1a;">
        <h3 style="margin-top:0; color: #2e7d32;">{html.escape(right_title)}</h3>
        <div style="font-size:13px; color:#444; margin-bottom:8px; border-bottom: 1px solid #c8e6c9; padding-bottom: 5px;"><b>Retrieved context:</b><br>{safe_context}</div>
        <div style="white-space:normal; line-height:1.5; font-weight: 400;">{safe_right}</div>
      </div>
    </div>
    ''') )


def compare_responses(user_query: str, use_rag: bool = False, top_k: int = RAG_TOP_K):
    context = retrieve_context(user_query, top_k=top_k) if use_rag else ""
    prompt = build_inference_prompt(user_query, context, use_rag=use_rag)

    base_model_eval = load_base_inference_model()
    base_answer = generate_response(base_model_eval, prompt)

    del base_model_eval
    gc.collect()
    torch.cuda.empty_cache()

    tuned_model_eval = load_tuned_inference_model()
    tuned_answer = generate_response(tuned_model_eval, prompt)

    del tuned_model_eval
    gc.collect()
    torch.cuda.empty_cache()

    mode_title = "Without RAG" if not use_rag else "With light RAG"
    _render_comparison_card(
        f"Evaluation mode: {mode_title}",
        context,
        "Base model" if not use_rag else "Base model + light RAG",
        base_answer,
        "Fine-tuned model" if not use_rag else "Fine-tuned model + light RAG",
        tuned_answer,
    )

    return base_answer, tuned_answer, context

import gc
import torch

# Note: This logic assumes 'trainer' exists from previous cells.
trainer.model.cpu()
del trainer
gc.collect()
torch.cuda.empty_cache()

sample_query = "My laptop is overheating and the support app keeps crashing after the update. What should I do?"
print("Evaluation 1: no RAG")
base_answer_no_rag, tuned_answer_no_rag, context_no_rag = compare_responses(sample_query, use_rag=False)
print("Evaluation 2: light RAG")
base_answer_rag, tuned_answer_rag, context_rag = compare_responses(sample_query, use_rag=True)

Evaluation 1: no RAG


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluation 2: light RAG


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## 10) Save everything to Google Drive

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import drive

mount_point = "/content/gdrive"
os.makedirs(mount_point, exist_ok=True)

drive.mount(mount_point)

ADAPTER_DIR = Path(mount_point) / "MyDrive" / "OmniSupport_Adapter_Weights"

if ADAPTER_DIR.exists():
    shutil.rmtree(ADAPTER_DIR)

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

train_model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

print("Adapter saved to:", ADAPTER_DIR)

Mounted at /content/gdrive
Adapter saved to: /content/gdrive/MyDrive/OmniSupport_Adapter_Weights


## 11) Try your own prompts

In [ ]:
user_query = "I cannot log into my account after resetting my password. What should I check?"
print("Evaluation 1: no RAG")
compare_responses(user_query, use_rag=False)
print("Evaluation 2: light RAG")
compare_responses(user_query, use_rag=True)


Evaluation 1: no RAG


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=180) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


("system\n\nYou are OmniSupport AI, a customer support assistant. Give clear, constructive, step-by-step help in a warm professional tone. Acknowledge the issue, use a consistent support structure, explain the next actions, and mention what to do if the problem persists. Keep the answer concise, practical, and easy to follow.user\n\nUse the retrieved support notes only as light context. Do not copy them verbatim.\n\nRetrieved support notes:\nCategory: account_access. Guidance: Account access troubleshooting: confirm the registered email, reset the password, check spam or junk folders for reset links, clear browser cache, disable VPN temporarily, and verify 2FA or backup codes before escalation.\n\nCustomer question:\nI cannot log into my account after resetting my password. What should I check?\n\nAnswer with a warm support tone, a short structured workflow, and practical next steps.assistant\n\nHi there!\n\nThank you for reaching out to us about your account access issue. I'm here to 